## ML_1021: Sliding Window Attention

**Difficulty**: Hard | **TorchCode**: #11

### Problem
Implement Sliding Window Attention (Mistral-style). Position `i` can only attend to positions `j` where `|i - j| ≤ window_size`. Mask all other positions with `-inf`.

### Formula
$$M_{ij} = \begin{cases} 0 & |i-j| \le w \\ -\infty & |i-j| > w \end{cases}$$

In [1]:
import torch
import torch.nn as nn
import math


class SlidingWindowGroupQueryAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int, window_size: int):
        super().__init__()
        assert d_model % num_heads == 0
        assert num_heads % num_kv_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.window_size = window_size
        self.d_k = d_model // num_heads
        self.repeats = num_heads // num_kv_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_o = nn.Linear(d_model, d_model)

    def repeat_kv(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, num_kv_heads, S, d_k)
        return: (B, num_heads, S, d_k)
        """
        B, H_kv, S, D = x.shape
        x = x.unsqueeze(2)                                  # (B, H_kv, 1, S, D)
        x = x.expand(B, H_kv, self.repeats, S, D)          # (B, H_kv, repeats, S, D)
        x = x.contiguous().view(B, H_kv * self.repeats, S, D)
        return x

    def softmax_manual(self, x: torch.Tensor, dim: int = -1) -> torch.Tensor:
        max_value = x.max(dim=dim, keepdim=True).values
        x = x - max_value
        exp_x = torch.exp(x)
        sum_exp = exp_x.sum(dim=dim, keepdim=True)
        return exp_x / sum_exp

    def build_sliding_window_mask(self, S: int, device) -> torch.Tensor:
        """
        mask[i, j] = True if token i can attend to token j
        Allowed:
          j <= i                  (causal)
          i - j < window_size     (inside window)
        """
        row_idx = torch.arange(S, device=device).unsqueeze(1)   # (S, 1)
        col_idx = torch.arange(S, device=device).unsqueeze(0)   # (1, S)

        causal_mask = col_idx <= row_idx
        window_mask = (row_idx - col_idx) < self.window_size

        mask = causal_mask & window_mask
        return mask  # (S, S)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, S, d_model)
        """
        B, S, _ = x.shape

        # 1. Project
        q = self.W_q(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(B, S, self.num_kv_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, S, self.num_kv_heads, self.d_k).transpose(1, 2)

        # 2. Expand KV heads
        k = self.repeat_kv(k)
        v = self.repeat_kv(v)

        # 3. Attention scores
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)   # (B, H, S, S)

        # 4. Sliding window causal mask
        mask = self.build_sliding_window_mask(S, x.device)         # (S, S)
        scores = scores.masked_fill(~mask, float("-inf"))

        # 5. Manual softmax
        attn = self.softmax_manual(scores, dim=-1)

        # 6. Weighted sum
        out = attn @ v                                             # (B, H, S, d_k)

        # 7. Merge heads
        out = out.transpose(1, 2).contiguous().view(B, S, self.d_model)

        # 8. Output projection
        out = self.W_o(out)
        return out

torch.manual_seed(0)

model = SlidingWindowGroupQueryAttention(
    d_model=32,
    num_heads=4,
    num_kv_heads=2,
    window_size=3
)

x = torch.randn(2, 6, 32)
out = model(x)
print(out.shape)   # should be torch.Size([2, 6, 32])


/home/kenneth/CODE/AI_PROJECTS/Leetcode/.venv/lib/python3.10/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


torch.Size([2, 6, 32])


In [ ]:
import torch
import math

def sliding_window_attention(Q, K, V, window_size):
    d_k = K.size(-1)
    scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(d_k)
    S = Q.size(1)
    idx = torch.arange(S, device=Q.device)
    mask = (idx.unsqueeze(0) - idx.unsqueeze(1)).abs() > window_size
    scores = scores.masked_fill(mask.unsqueeze(0), float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    return torch.bmm(weights, V)

In [ ]:
import torch
import math

# --- Test 1: Output shape ---
out = sliding_window_attention(torch.randn(2, 8, 16), torch.randn(2, 8, 16), torch.randn(2, 8, 16), window_size=2)
assert out.shape == (2, 8, 16)

# --- Test 2: window_size=0 — only sees itself ---
Q = torch.randn(1, 4, 8); K = torch.randn(1, 4, 8); V = torch.randn(1, 4, 8)
out = sliding_window_attention(Q, K, V, window_size=0)
assert torch.allclose(out, V, atol=1e-5)

# --- Test 3: Large window equals full attention ---
torch.manual_seed(0)
B, S, D = 2, 6, 8
Q = torch.randn(B, S, D); K = torch.randn(B, S, D); V = torch.randn(B, S, D)
out_win = sliding_window_attention(Q, K, V, window_size=S)
scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(D)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
assert torch.allclose(out_win, ref, atol=1e-5)

# --- Test 4: Distant tokens don't affect output ---
torch.manual_seed(0)
B, S, D = 1, 10, 8
Q = torch.randn(B, S, D); K = torch.randn(B, S, D); V = torch.randn(B, S, D)
out1 = sliding_window_attention(Q, K, V, window_size=1)
K2, V2 = K.clone(), V.clone()
K2[:, 5:] = torch.randn(B, 5, D); V2[:, 5:] = torch.randn(B, 5, D)
out2 = sliding_window_attention(Q, K2, V2, window_size=1)
assert torch.allclose(out1[:, 0], out2[:, 0], atol=1e-5)

# --- Test 5: Gradient flow ---
Q = torch.randn(2, 4, 8, requires_grad=True)
K = torch.randn(2, 4, 8, requires_grad=True)
V = torch.randn(2, 4, 8, requires_grad=True)
sliding_window_attention(Q, K, V, window_size=1).sum().backward()
assert Q.grad is not None

print('All tests passed!')